## MISA (2024-2025)
- Alohan'ny mamerina dia avereno atao Run ny notebook iray manontolo. Ny fanaovana azy dia redémarrena mihitsy ny kernel aloha (jereo menubar, safidio **Kernel$\rightarrow$Restart Kernel and Run All Cells**).

- Izay misy hoe `YOUR CODE HERE` na `YOUR ANSWER HERE` ihany no fenoina. Afaka manampy cells vaovao raha ilaina. Aza adino ny mameno references eo ambany raha ilaina.

## References
Eto ilay references rehetra no apetraka
* Nampiasa chatgpt

---

In [1]:
from sklearn.decomposition  import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_iris, load_breast_cancer, make_blobs
import numpy as np
from random import randrange
from sklearn.metrics import accuracy_score
import cvxpy as cp

def grad_check_sparse(f, x, analytic_grad, num_checks=12, h=1e-5, error=1e-9):
    """
    sample a few random elements and only return numerical
    in this dimensions
    """

    for i in range(num_checks):
        ix = tuple([randrange(m) for m in x.shape])

        oldval = x[ix]
        x[ix] = oldval + h  # increment by h
        fxph = f(x)  # evaluate f(x + h)
        x[ix] = oldval - h  # increment by h
        fxmh = f(x)  # evaluate f(x - h)
        x[ix] = oldval  # reset

        grad_numerical = (fxph - fxmh) / (2 * h)
        grad_analytic = analytic_grad[ix]
        rel_error = abs(grad_numerical - grad_analytic) / (
            abs(grad_numerical) + abs(grad_analytic)
        )
        print(
            "numerical: %f analytic: %f, relative error: %e"
            % (grad_numerical, grad_analytic, rel_error)
        )
        assert rel_error < error

def rel_error(x, y):
    """ returns relative error """
    return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

data = load_iris()
X, y = data.data, data.target

# PCA

In [229]:
class PrincipalComponentAnalysis():
    def __init__(self, n_components):
        self.n_components = n_components
        self.components = None
    
    def fit(self, X):
        # YOUR CODE HERE
        # SVD
        U, S, Vt = np.linalg.svd(X, full_matrices=False)
        self.components = Vt[:self.n_components].T

    def transform(self, X):
        # YOUR CODE HERE
        return X_centered @ self.components

In [230]:
X_centered = X - np.mean(X, axis=0)

pca = PCA(n_components=3)
pca.fit(X_centered)
X_pca_trans = pca.transform(X_centered)

model = PrincipalComponentAnalysis(n_components=3)
model.fit(X_centered)
X_model_trans = model.transform(X_centered)

sign_correct_X_model_trans = np.concatenate([np.expand_dims(X_model_trans[:,0],axis=1),-X_model_trans[:,1:]],axis=1)

error = rel_error(X_pca_trans, sign_correct_X_model_trans)
print(error)
assert  error < 1e-11

1.5769484752555976e-13


# Binary Linear SVM with CVXPY

## Hard margin 

In [231]:
X2, y2 = make_blobs(n_samples=300, centers=2, n_features=12, random_state=47)
scaler = StandardScaler()
X2 = scaler.fit_transform(X2)
y2[y2 == 0] = -1

$$\min_{\mathbf{w},b}\frac{1}{2}||\mathbf{w}||^2$$ <br>
$$\text{s.t } y_i(\mathbf{w}^{\top}\mathbf{x}_i + b) \ge 1, \ i=1...N$$

In [133]:
class LinearSVMHardMargin():
    def __init__(self):
        self.w = None
        self.b = 0
    
    def fit(self, X, y):
        # YOUR CODE HERE
        w =  cp.Variable(X.shape[1])
        b = cp.Variable()

        # Problem
        objective = cp.Minimize(0.5 * cp.norm1(w)**2)
        constraint = cp.multiply(y, X @ w + b) >= 1
        problem = cp.Problem(objective, [constraint])
        
        problem.solve()
        self.w = w.value
        self.b = b.value
        
    def predict(self, X):
        """Return the predicted label 1 or -1"""        
        y_pred = X @ self.w + self.b
        return np.sign(y_pred)

In [134]:
model = LinearSVMHardMargin()
model.fit(X2, y2)
pred = model.predict(X2)
accuracy = accuracy_score(y2, pred)
print(accuracy)
assert accuracy == 1

1.0


/tmp/ipykernel_6327/3639633432.py:16: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  problem.solve()


## Soft Margin

In [135]:
data3 = load_breast_cancer()
X3, y3 = data3.data, data3.target
scaler = StandardScaler()
X3 = scaler.fit_transform(X3)
y3[y3 == 0] = -1

$$L(\mathbf{w},b) = \frac{1}{N} \sum_{i=1}^N \max(0, y_i(\mathbf{w}^{\top}\mathbf{x}_i + b)) + \lambda||\mathbf{w}||^2_2$$

In [136]:
class LinearSVMSoftMargin():
    def __init__(self, alpha=0):
        self.w = None
        self.b = 0
        self.alpha = alpha
    
    def fit(self, X, y):
        # YOUR CODE HERE
        w =  cp.Variable(X.shape[1])
        b = cp.Variable()
        
        # Problem
        HL = cp.sum(cp.pos(1 - cp.multiply(y, X @ w + b))) / X.shape[0]
        reg = cp.sum_squares(w)
        objective = cp.Minimize(HL + self.alpha * reg)
        problem = cp.Problem(objective)
        
        problem.solve()
        self.w = w.value
        self.b = b.value
        
    def predict(self, X):
        """Return the predicted label 1 or -1"""
        # YOUR CODE HERE
        y_pred = X @ self.w + self.b
        return np.sign(y_pred)

In [137]:
model = LinearSVMSoftMargin(alpha=1e-3)
model.fit(X3, y3)
pred = model.predict(X3)
accuracy = accuracy_score(y3, pred)
print(accuracy)
assert accuracy >= 0.98

0.9876977152899824


# Multiclass Linear SVM

## Loss

$$L(\mathbf{W}) = \sum_{i=1}^N \sum_{j \neq y_i} \max(0, s_j - s_{y_i} + 1) + \lambda||\mathbf{w}||^2_2$$ <br>
$$\text{where } s_j = (f(\mathbf{x}_i;\mathbf{W}))_j = (\mathbf{W}\mathbf{x}_i)_j \text{ is the score for the j-th class}$$

In [138]:
W = np.random.randn(X.shape[1], 3) * 0.0001

In [166]:
def svm_loss_naive(W, X, y, alpha):
    """
    Multiclass SVM loss function WITH FOR LOOPS

    Inputs:
    - W: array of shape (D, C) containing weights
    - X: array of shape (N, D) containing a minibatch of data
    - y: array of shape (N,) containing training labels
    - alpha: (float) regularization 

    Returns a tuple of:
    - loss as single float
    - gradient with respect to weights W;  same shape as W
    """
    
    # Initialization
    loss = 0.0
    dW = np.zeros_like(W)
    
    # YOUR CODE HERE
    N, D = X.shape
    C = W.shape[1]
    loss = 0.0
    dW = np.zeros_like(W)

    for i in range(N):
        scores = X[i] @ W  # (C,)
        M = 0

        for j in range(C):
            margin = scores[j] - scores[y[i]] + 1
            if margin > 0:
                loss += margin
                # Gradient for incorrect class
                dW[:, j] += X[i]
                M += 1

        # Gradient for correct class
        dW[:, y[i]] -= M * X[i]

    # Regularization
    loss += alpha * np.sum(W * W)
    dW += 2 * alpha * W
    
    return loss, dW

In [167]:
# NO REGLARIZATION
loss, dW = svm_loss_naive(W, X, y, 0.0)

f = lambda W: svm_loss_naive(W, X, y, 0.0)[0]
grad_numerical = grad_check_sparse(f, W, dW, error=1e-9)

numerical: -75.300000 analytic: -75.300000, relative error: 7.342152e-11
numerical: -124.000000 analytic: -124.000000, relative error: 2.118208e-11
numerical: -75.300000 analytic: -75.300000, relative error: 7.342152e-11
numerical: 12.500000 analytic: 12.500000, relative error: 4.665716e-10
numerical: 125.600000 analytic: 125.600000, relative error: 6.305478e-11
numerical: -111.700000 analytic: -111.700000, relative error: 5.375571e-11
numerical: -19.000000 analytic: -19.000000, relative error: 3.648530e-10
numerical: -19.000000 analytic: -19.000000, relative error: 3.648530e-10
numerical: 43.100000 analytic: 43.100000, relative error: 1.554091e-10
numerical: 143.000000 analytic: 143.000000, relative error: 4.004639e-11
numerical: -19.000000 analytic: -19.000000, relative error: 3.648530e-10
numerical: -269.100000 analytic: -269.100000, relative error: 1.031399e-11


In [168]:
#REGLARIZATION
loss, dW = svm_loss_naive(W, X, y, 2)

f = lambda W: svm_loss_naive(W, X, y, 2)[0]
grad_numerical = grad_check_sparse(f, W, dW, error=1e-9)

numerical: -269.099488 analytic: -269.099488, relative error: 1.223171e-11
numerical: -13.899987 analytic: -13.899987, relative error: 1.442328e-10
numerical: 344.399816 analytic: 344.399816, relative error: 4.492184e-12
numerical: -269.099488 analytic: -269.099488, relative error: 1.223171e-11
numerical: 12.500097 analytic: 12.500097, relative error: 4.126240e-10
numerical: 125.600825 analytic: 125.600825, relative error: 6.799289e-11
numerical: -13.899987 analytic: -13.899987, relative error: 1.442328e-10
numerical: 43.099902 analytic: 43.099902, relative error: 1.484083e-10
numerical: -269.099488 analytic: -269.099488, relative error: 1.223171e-11
numerical: -123.999307 analytic: -123.999307, relative error: 2.215410e-11
numerical: -13.899987 analytic: -13.899987, relative error: 1.442328e-10
numerical: -55.600507 analytic: -55.600507, relative error: 2.500429e-11


In [224]:
def svm_loss_vectorized(W, X, y, alpha):
    """
    Multiclass SVM loss function WITHOUT FOR LOOPS

    Inputs:
    - W: array of shape (D, C) containing weights
    - X: array of shape (N, D) containing a minibatch of data
    - y: array of shape (N,) containing training labels
    - alpha: (float) regularization 

    Returns a tuple of:
    - loss as single float
    - gradient with respect to weights W;  same shape as W
    """
    # Initialize the loss and gradient to zero.
    loss = 0.0
    dW = np.zeros_like(W)

    # YOUR CODE HERE
    N = X.shape[0]

    # Scores
    scores = X @ W  # (N, C)
    correct_class_scores = scores[np.arange(N), y][:, np.newaxis] # (N,1)
    
    # Margins
    margins = np.maximum(0, scores - correct_class_scores + 1)
    margins[np.arange(N), y] = 0  # Set 0 for correct class
    
    # Loss
    loss = np.sum(margins)
    loss += alpha * np.sum(W*W)

    # Gradient
    mask = (margins > 0).astype(float)  # (N, C)
    M = np.sum(mask, axis=1)  # (N,)

    # For correct class 
    mask[np.arange(N), y] = -M

    dW = X.T @ mask  # (D, C)
    dW += 2 * alpha * W


    return loss, dW

In [225]:
# NO REGLARIZATION
loss, dW = svm_loss_vectorized(W, X, y, 0.0)

f = lambda W: svm_loss_vectorized(W, X, y, 0.0)[0]
grad_numerical = grad_check_sparse(f, W, dW, error=1e-9)

numerical: -111.700000 analytic: -111.700000, relative error: 2.866153e-12
numerical: 12.500000 analytic: 12.500000, relative error: 1.018615e-10
numerical: -75.300000 analytic: -75.300000, relative error: 2.068123e-12
numerical: 12.500000 analytic: 12.500000, relative error: 1.018615e-10
numerical: 12.500000 analytic: 12.500000, relative error: 1.018615e-10
numerical: 43.100000 analytic: 43.100000, relative error: 2.352168e-11
numerical: 43.100000 analytic: 43.100000, relative error: 2.352168e-11
numerical: 125.600000 analytic: 125.600000, relative error: 4.831012e-12
numerical: -111.700000 analytic: -111.700000, relative error: 2.866153e-12
numerical: -111.700000 analytic: -111.700000, relative error: 2.866153e-12
numerical: 43.100000 analytic: 43.100000, relative error: 2.352168e-11
numerical: -75.300000 analytic: -75.300000, relative error: 2.068123e-12


In [226]:
# REGLARIZATION
loss, dW = svm_loss_vectorized(W, X, y, 2)

f = lambda W: svm_loss_vectorized(W, X, y, 2)[0]
grad_numerical = grad_check_sparse(f, W, dW, error=1e-9)

numerical: 43.099902 analytic: 43.099902, relative error: 1.652057e-11
numerical: 344.399816 analytic: 344.399816, relative error: 3.659173e-13
numerical: -111.699638 analytic: -111.699638, relative error: 2.761647e-12
numerical: -18.999742 analytic: -18.999742, relative error: 6.612410e-11
numerical: -13.899987 analytic: -13.899987, relative error: 4.198813e-11
numerical: -55.600507 analytic: -55.600507, relative error: 2.611355e-11
numerical: 143.000101 analytic: 143.000101, relative error: 1.336475e-11
numerical: -111.699638 analytic: -111.699638, relative error: 2.761647e-12
numerical: 125.600825 analytic: 125.600825, relative error: 1.075424e-13
numerical: 344.399816 analytic: 344.399816, relative error: 3.659173e-13
numerical: -123.999307 analytic: -123.999307, relative error: 7.663017e-13
numerical: -18.999742 analytic: -18.999742, relative error: 6.612410e-11


## Gradient descent

In [227]:
class LinearModel():
    def __init__(self, fit_intercept=True):
        self.W = None
        self.fit_intercept = fit_intercept

    def train(self, X, y, learning_rate=1e-3, alpha=0, num_iters=100, batch_size=200, verbose=False):
        if self.fit_intercept:
            # YOUR CODE HERE
            X = np.insert(X, 0, values=1, axis=1)
            
        N, d = X.shape
        
        C = (np.max(y) + 1) 
        if self.W is None: # Initialization
            self.W = 0.001 * np.random.randn(d, C)

        # Run stochastic gradient descent to optimize W
        
        loss_history = []
        for it in range(num_iters):
            X_batch = None
            y_batch = None
                                                               
            # Sample batch_size elements in X_batch and y_batch
            # X_batch shape is  (batch_size, d) and y_batch shape is (batch_size,)                                                                                          
            # Hint: Use np.random.choice to generate indices
            # YOUR CODE HERE
            idx = np.random.choice(X.shape[0], batch_size, replace=False)
            X_batch = X[idx]
            y_batch = y[idx]
            
            # evaluate loss and gradient
            loss, dW = self.loss(X_batch, y_batch, alpha)
            loss_history.append(loss)

            # perform parameter update                                                                
            # Update the weights w using the gradient and the learning rate.          
            # YOUR CODE HERE
            self.W -= learning_rate * dW
            
            if verbose and it % 10000 == 0:
                print("iteration %d / %d: loss %f" % (it, num_iters, loss))
                
        return loss_history

    def predict(self, X):
        pass

    def loss(self, X_batch, y_batch, reg):
        pass

class LinearSVM(LinearModel):
    """ Softmax regression """

    def loss(self, X_batch, y_batch, alpha):
        return svm_loss_vectorized(self.W, X_batch, y_batch, alpha)
    
    def predict(self, X):
        """ 
        Inputs:
        - X: array of shape (N, D) 

        Returns:
        - y_pred: 1-dimensional array of length N, each element is an integer giving the predicted class 
        """
        # YOUR CODE HERE
        if self.fit_intercept:
            X = np.insert(X, 0, values=1, axis=1)
        scores = X @ self.W
        y_pred = np.argmax(scores, axis=1)
        return y_pred

In [228]:
model = LinearSVM(fit_intercept=True)
model.train(X, y, num_iters=75000, batch_size=64, learning_rate=1e-3, verbose=True)
pred = model.predict(X)
model_accuracy = accuracy_score(y, pred)
print(model_accuracy)
assert model_accuracy > 0.97

iteration 0 / 75000: loss 128.176180
iteration 10000 / 75000: loss 0.826294
iteration 20000 / 75000: loss 6.951236
iteration 30000 / 75000: loss 6.090466
iteration 40000 / 75000: loss 1.216871
iteration 50000 / 75000: loss 2.673776
iteration 60000 / 75000: loss 0.589596
iteration 70000 / 75000: loss 5.462587
0.98
